In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA

def preprocessing_data(dataset):

    ## Data Cleaning

    # Eliminate missing values
    dataset.isnull().sum()
    dataset = dataset.dropna()

    # Eliminate duplicate values
    dataset.duplicated().sum() # no duplicated values

    # Eliminate inconsistent values
    dataset.drop(dataset[dataset['Exam_Score'] > 100].index , inplace = True)
    dataset.drop(dataset[dataset['Exam_Score'] < 0].index , inplace = True)
    dataset.drop(dataset[dataset['Attendance'] > 100].index , inplace = True)
    dataset.drop(dataset[dataset['Attendance'] < 0].index , inplace = True)
    dataset.drop(dataset[dataset['Previous_Scores'] > 100].index , inplace = True)
    dataset.drop(dataset[dataset['Previous_Scores'] < 0].index , inplace = True)

    # reset index
    dataset.reset_index(drop=True, inplace=True)

    ## Data Transformation

    categorical_features = ['Extracurricular_Activities', 'Internet_Access', 'School_Type', 'Learning_Disabilities', 'Gender', 'Parental_Involvement', 'Access_to_Resources', 'Motivation_Level', 'Family_Income', 'Teacher_Quality', 'Peer_Influence', 'Parental_Education_Level', 'Distance_from_Home']

    continuous_features = ['Hours_Studied', 'Attendance', 'Previous_Scores', 'Sleep_Hours', 'Tutoring_Sessions', 'Physical_Activity', 'Exam_Score']

    # encoding categorical features

    encoded_categorical_features = pd.get_dummies(dataset[categorical_features], drop_first=False)
    print(encoded_categorical_features.head())

    # Create processed_dataset by dropping original categorical columns and concatenating one-hot encoded ones

    processed_dataset = dataset.drop(columns=categorical_features).copy()
    processed_dataset = pd.concat([processed_dataset, encoded_categorical_features], axis=1)

    processed_dataset.to_csv('processed_data.csv', index = False)

    ## Sampling
    features_column = [f for f in encoded_categorical_features.columns] + [f for f in continuous_features]

    # Ensure that all columns in features_column exist in processed_dataset
    missing_cols = [col for col in features_column if col not in processed_dataset.columns]
    if missing_cols:
        raise KeyError(f"The following columns are missing from processed_dataset: {missing_cols}")
     

    ## Normalize the numerical features
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(processed_dataset[features_column].values)

    return features_scaled, processed_dataset, continuous_features


def K_Means_Clustering(features_scaled, preprocessed_data):
    # --- K-Means Clustering ---

    # Determine the optimal number of clusters using the Elbow Method

    inertia = []
    k_range = range(1, 11)
    for k in k_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(features_scaled)
        inertia.append(kmeans.inertia_)

    # Perform clustering 

    optimal_k = 3
    kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    kmeans_labels = kmeans.fit_predict(features_scaled)
    preprocessed_data['KMeans_Cluster'] = kmeans_labels

    return preprocessed_data

def K_Means_visualization(features_scaled, preprocessed_data):
    # Visualize the clusters

    # Apply PCA for dimensionality reduction to 3 components
    pca = PCA(n_components=3)
    principal_components = pca.fit_transform(features_scaled)

    # Create a DataFrame for plotting
    pca_df = pd.DataFrame(data=principal_components, columns=['principal_component_1', 'principal_component_2', 'principal_component_3'])
    pca_df['KMeans_Cluster'] = preprocessed_data['KMeans_Cluster']

    # Create the 3D scatter plot
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Get unique cluster labels and a color map
    unique_clusters = pca_df['KMeans_Cluster'].unique()
    colors = plt.cm.viridis(np.linspace(0, 1, len(unique_clusters)))

    # Plot each cluster separately to get distinct colors and a legend
    for i, cluster_id in enumerate(unique_clusters):
        cluster_data = pca_df[pca_df['KMeans_Cluster'] == cluster_id]
        ax.scatter(
            cluster_data['principal_component_1'],
            cluster_data['principal_component_2'],
            cluster_data['principal_component_3'],
            color=colors[i],
            s=100,
            alpha=0.7,
            label=f'Cluster {cluster_id}'
        )

    ax.set_title('K-Means Clusters Visualized with PCA (3 Components)')
    ax.set_xlabel(f'Principal Component 1 (Explained Variance: {pca.explained_variance_ratio_[0]:.2f})')
    ax.set_ylabel(f'Principal Component 2 (Explained Variance: {pca.explained_variance_ratio_[1]:.2f})')
    ax.set_zlabel(f'Principal Component 3 (Explained Variance: {pca.explained_variance_ratio_[2]:.2f})')
    ax.legend()
    plt.tight_layout()

    plt.show()

# def DBScan_Clustering(features_scaled, preprocessed_data): 
#     # --- DBScan Clustering ---

#     # Determine optimal eps using K-distance graph
#     # A common starting point for min_samples is 2 * number_of_dimensions,
#     # or often 5-10 for higher dimensional data.
#     # Let's use min_samples = 5 as a reasonable default.
#     min_samples_val = 5
#     neigh = NearestNeighbors(n_neighbors=min_samples_val)
#     nbrs = neigh.fit(features_scaled)
#     distances, indices = nbrs.kneighbors(features_scaled)

#     # Sort distances and plot the K-distance graph
#     # This helps in visually determining the optimal 'eps' value (the elbow point)
#     distances = np.sort(distances[:, min_samples_val-1], axis=0)
#     plt.figure(figsize=(10, 6))
#     plt.plot(distances)
#     plt.title(f'K-distance Graph for DBSCAN (k={min_samples_val})')
#     plt.xlabel('Data Points sorted by distance')
#     plt.ylabel(f'{min_samples_val}-th nearest neighbor distance')
#     plt.grid(True)
#     plt.savefig('dbscan_k_distance_graph.png') # Save plot for review
#     plt.close() # Close plot

#     # Perform clustering
#     # For demonstration, we'll pick an 'eps' value.
#     # In a real scenario, you would inspect 'dbscan_k_distance_graph.png'
#     # and choose the 'elbow' point for optimal 'eps'.
#     # A common range for scaled data might be 0.5 to 1.0. Let's start with 0.7.
#     optimal_eps = 0.7 # This value is typically determined by inspecting the K-distance graph

#     dbscan = DBSCAN(eps=optimal_eps, min_samples=min_samples_val)
#     dbscan_labels = dbscan.fit_predict(features_scaled)
#     preprocessed_data['DBSCAN_Cluster'] = dbscan_labels

#     return preprocessed_data

# def DBScan_visualization(features_scaled, preprocessed_data):
#     # Visualize the clusters
#     print('Hello World')



if _name_ == "_main_":
    
    dataset = pd.read_csv('Student Performance.csv')
    features_scaled, preprocessed_data, continuous_features = preprocessing_data(dataset)

    # K-Means-Clustering
    K_Means_Clustering(features_scaled, preprocessed_data)
    K_Means_visualization(features_scaled, preprocessed_data)

    # DBScan-Clustering
    DBScan_Clustering(features_scaled, preprocessed_data)
    DBScan_visualization(features_scaled, preprocessed_data)

NameError: name '_name_' is not defined